# Chapter 7: Tool Integration

Estimated time: ~7 hours.

Prerequisites: Chapter 1 (`agentlib.llm_client`), Chapter 4 (retry with backoff).

## Setup

The MCP server this chapter calls is real: a genuine tool exposing real package metadata
from PyPI's JSON API, not a mock. The server script lives at
`curriculum/_ch07_mcp_server.py` and is spawned as a subprocess over stdio transport.

In [1]:
import json
import os
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import BaseModel, ValidationError

from agentlib import llm_client
from agentlib.grading import check

_MCP_SERVER_PATH = str(_repo_root / "curriculum" / "_ch07_mcp_server.py")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print(f"MCP server script: {_MCP_SERVER_PATH}")
print("This chapter makes no model calls: the work is against a real MCP server over a real\n"
      "stdio transport, which is a tool integration rather than a model one. HAS_KEY is\n"
      "printed for consistency with the other chapters, not because anything here\n"
      "branches on it.")

LLM_PROVIDER = 'anthropic', HAS_KEY = False
MCP server script: /home/user/learning-agentic-ai/curriculum/_ch07_mcp_server.py
This chapter makes no model calls: the work is against a real MCP server over a real
stdio transport, which is a tool integration rather than a model one. HAS_KEY is
printed for consistency with the other chapters, not because anything here
branches on it.


## Section 1: Definitions

### MCP (Model Context Protocol)

**MCP** (Anthropic, 2024) standardizes how agent frameworks connect to external tools.
Without it, every framework writes custom glue for every tool: M frameworks times N tools
equals M*N integrations. MCP collapses that to M+N. Any MCP-compliant client talks to any
MCP-compliant server the same way any HTTP client talks to any HTTP server.

A server exposes a set of **tools**, each with a name, description, and a JSON Schema for
its inputs. A client discovers those tools at connection time and calls them by name with
structured arguments.

Real-world examples:
- **Claude Desktop** connects to local MCP servers (file system, database, custom tools)
  over stdio transport.
- **VS Code MCP integration** (2024): editor plugins expose coding tools as MCP servers.
- **Anthropic MCP specification** (2024): the protocol itself, defining client-server
  handshake, tool discovery, and JSON-RPC messaging.

### Schema validation

A tool's declared input/output schema is a **contract**, not documentation. If the tool
says it returns `{"name": str, "version": str}`, the caller should verify that before
acting on it. Pydantic is the standard library for this in Python.

Real-world examples:
- **FastAPI** uses Pydantic to validate every request and response.
- **OpenAPI specs** define JSON Schema contracts for HTTP APIs.
- **Pydantic in MCP servers**: the `mcp` Python package uses Pydantic for tool schemas.

### Four-part failure taxonomy

Not every tool failure needs the same response. Four distinct failure shapes:

| Failure type | What it looks like | Correct response |
|---|---|---|
| **Transient** | Timeout, connection reset, rate limit | Retry with backoff |
| **Malformed** | Response does not parse or type-check | Switch to alternate source |
| **Semantically wrong** | Valid shape, wrong content | Ask a human or verification step |
| **Version mismatch** | A required field is missing entirely | Ask a developer to update |

Treating them identically (retry everything, or silently swallow everything) is a common
production mistake. A malformed response retried against the same corrupted source returns
the same corruption. A semantically wrong response retried returns the same wrong data.

### Structured output strategies

Three ways to handle a model that returns broken JSON:

1. **Repair afterwards**: strip markdown fences and prose, find the braces, parse. Cheap,
   needs no cooperation from the model. Cannot fix broken syntax inside the JSON.
2. **Retry until it parses**: re-prompt on failure, up to a budget. Works for transient
   failures; wastes the budget on systematic ones (same prompt, same model, same break).
3. **Constrained decoding**: at each step, discard every candidate token that could not
   appear at this position in a valid JSON instance. The output is well-formed by
   construction. One generation is always enough.

Real-world examples:
- **OpenAI structured outputs** (`response_format`): constrained decoding.
- **llama.cpp GBNF grammars**: constrained decoding at the inference level.
- **Outlines** (dottxt): regex-to-FSM compiler for grammar-masked generation.

## Section 2: Concept Explanation

### MCP stdio transport

```
  Agent (client)                         Tool (server)
  +------------------+                   +------------------+
  |                  |   spawn process   |                  |
  |  1. Launch       | ----------------> | _ch07_mcp_server |
  |                  |                   |                  |
  |  2. Initialize   | --- JSON-RPC ---> | Handshake: agree |
  |     handshake    | <-- JSON-RPC ---- | on capabilities  |
  |                  |                   |                  |
  |  3. call_tool    | --- JSON-RPC ---> | Execute tool,    |
  |     (name, args) | <-- JSON-RPC ---- | return result    |
  |                  |                   |                  |
  |  4. Close        | --- stdin EOF --> | Process exits    |
  +------------------+                   +------------------+

  Transport: stdin/stdout pipe (no network, no ports)
  Protocol:  JSON-RPC 2.0 messages
```

The ordering matters: `session.initialize()` must complete before any `call_tool`.
A tool call issued before the handshake finishes hangs or returns a protocol error
several frames removed from the actual mistake.

### Failure classification decision tree

```
  Raw tool response arrives
       |
       v
  Did the call itself fail?
  (timeout, connection reset, rate limit)
       |
      YES --> TRANSIENT --> retry with backoff
       |
      NO
       |
       v
  Does it pass schema validation?
       |
      NO --> Is a required field MISSING entirely?
       |          |
       |         YES --> VERSION MISMATCH --> ask developer
       |          |
       |         NO --> MALFORMED --> switch source / alternate path
       |
      YES
       |
       v
  Does the content match what was requested?
  (e.g., asked for "numpy", got data for "numpy"?)
       |
      NO --> SEMANTICALLY WRONG --> ask human / verification
       |
      YES --> OK --> proceed
```

### Trade-offs: repair vs. retry vs. constrained decoding

| Strategy | Cost | Reliability | When it fails |
|---|---|---|---|
| Repair | 1 generation | Fixes wrapping, not internal syntax | Unquoted keys, truncated objects |
| Retry | 1-N generations | Works for transient failures | Systematic failures repeat identically |
| Constrained | 1 generation | Guarantees valid shape | Does not guarantee correct content |

A common interview mistake: claiming constrained decoding "solves" structured output.
It guarantees the output is **well-formed and on-schema**. It guarantees nothing about
the content being **correct**. A grammar mask that forces valid JSON can still produce
`{"name": "nump"}` -- perfect shape, wrong data.

## Section 3: Example Code Segments

The Pydantic schema, the MCP server, and the structured-output mock model.

### The Pydantic schema contract

`PackageInfo` defines what the MCP server's `get_package_info` tool should return.
Optional fields (`license`, `home_page`, `project_urls`) are genuinely optional in real
PyPI data. Required fields (`name`, `version`, `summary`) should always be present for
a real package.

In [2]:
class PackageInfo(BaseModel):
    '''The schema contract for what this tool should return. Optional fields really are
    optional in real PyPI data -- e.g. many packages leave the legacy `home_page` field
    empty and only populate `project_urls` -- but name/version/summary should always be
    present for a real package.'''
    name: str
    version: str
    summary: str
    license: str | None = None
    home_page: str | None = None
    project_urls: dict | None = None


schema_path = _repo_root / "data" / "schemas" / "package_info.schema.json"
schema_path.write_text(json.dumps(PackageInfo.model_json_schema(), indent=2) + "\n")
print(f"Wrote schema to {schema_path.relative_to(_repo_root)}")
print(json.dumps(PackageInfo.model_json_schema(), indent=2))


Wrote schema to data/schemas/package_info.schema.json
{
  "description": "The schema contract for what this tool should return. Optional fields really are\noptional in real PyPI data -- e.g. many packages leave the legacy `home_page` field\nempty and only populate `project_urls` -- but name/version/summary should always be\npresent for a real package.",
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "version": {
      "title": "Version",
      "type": "string"
    },
    "summary": {
      "title": "Summary",
      "type": "string"
    },
    "license": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "License"
    },
    "home_page": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Home Page"
    },
    "project_urls": {
     

### The MCP server

`curriculum/_ch07_mcp_server.py` is a companion file, not notebook code. MCP's stdio
transport spawns the server as a separate process, so it has to exist as a real script
the client can launch. Its one tool, `get_package_info(package_name)`, reads from a
small local cache of real PyPI data (`data/pypi_cache/`, three packages: `requests`,
`numpy`, `anthropic`) and falls back to a live PyPI query for anything not cached.

### The structured-output mock model

`MockModel` from `agentlib.structured_outputs` generates JSON with configurable failure
modes. It fails the way real models fail: markdown fences, prose preamble, trailing
commas, unquoted keys. One detail matters: the correct token is always somewhere in its
candidate list, just not always the highest-scoring one. That is a real property of real
models, and it is the entire reason constrained decoding works.

In [13]:
from agentlib.structured_outputs import FIELDS, QUIRKS, RECORDS, MockModel

# The same record, from the same model, under three different sets of failure modes.
for label, quirks in [
    ("clean", []),
    ("wrapped in prose and fences", ["fence", "preamble", "epilogue"]),
    ("wrapped AND syntactically broken", QUIRKS),
]:
    raw = MockModel(RECORDS[0], quirks=quirks).generate()
    print(f"--- {label} ---")
    print(raw)
    print()

--- clean ---
{"name":"numpy","version":"2.4.6","summary":"Fundamental package for array computing"}

--- wrapped in prose and fences ---
Here is the JSON you asked for:
```json
{"name":"numpy","version":"2.4.6","summary":"Fundamental package for array computing"}

Let me know if you need anything else!

--- wrapped AND syntactically broken ---
Here is the JSON you asked for:
```json
{name:"numpy",version:"2.4.6",summary:"Fundamental package for array computing",}

Let me know if you need anything else!



## Section 4: Build It Yourself

Six graded tasks. The first two handle data arriving FROM a tool (schema validation,
MCP client). The last three handle data arriving FROM a model (JSON repair, retry with
budget, constrained decoding). The sixth (failure classifier) is in Break It below.

### Task 1: `validate_tool_output` (schema validation)

`model_validate` raises on bad data, so somewhere there has to be a `try`. What the
`except` block returns is one of the highest-stakes decisions in this chapter.

The tempting version catches `ValidationError` and returns `None`. But `None` from a
lookup already means "no such package." Collapse "the tool returned garbage" into that
same value and a broken integration becomes indistinguishable from an empty result. The
agent reports "I could not find anything about numpy" with total confidence and nobody
discovers the real problem for weeks.

In [ ]:
def validate_tool_output(response: dict, model):
    '''Validate a raw tool response against its schema.

    Return a (validated, error) pair: (the parsed model, None) on success, or
    (None, the ValidationError) on failure. Exactly one of the two is ever populated.

    Return the ValidationError object itself, not a flattened string -- the caller needs
    .errors() to tell a field that is missing entirely from one that is present with the
    wrong type, and those two mean very different things about what broke upstream.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


validate_tool_output = check("ch07-schema-validate", validate_tool_output)

### Task 2: `call_mcp_tool` (MCP client over stdio)

Four things have to happen in order: spawn the server over stdio, open a session on its
read/write streams, **complete the handshake**, and only then call a tool.

MCP is stateful. `session.initialize()` is where client and server negotiate what the
connection can do, and a `call_tool` issued before that finishes has nothing to route.

In [ ]:
async def call_mcp_tool(server_path: str, tool_name: str, arguments: dict) -> dict:
    '''Spawn the real MCP server, call one of its tools, return the raw (unvalidated) result.

    Use StdioServerParameters(command=sys.executable, args=[server_path]) with stdio_client,
    open a ClientSession on the (read, write) pair, await session.initialize() BEFORE any
    call_tool, then return the parsed JSON payload as a dict -- MCP hands back content blocks
    whose .text is a JSON string, so callers should not have to json.loads() it themselves.

    Pass errlog=devnull to stdio_client. That works around an environment quirk rather than
    teaching anything: under some headless notebook runners the kernel's stdout/stderr have
    no real file descriptor, which the subprocess machinery needs for the child's stderr.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


call_mcp_tool = check("ch07-mcp-client", call_mcp_tool)

#### Testing the client against the real server

In [5]:
async def call_get_package_info(package_name: str) -> dict:
    return await call_mcp_tool(_MCP_SERVER_PATH, "get_package_info", {"package_name": package_name})


raw_result = await call_get_package_info("requests")
print("Raw tool result:", raw_result)

validated = PackageInfo.model_validate(raw_result)
print(f"\nValidated: {validated.name} v{validated.version} -- {validated.summary!r}")

Raw tool result: {'name': 'requests', 'version': '2.34.2', 'summary': 'Python HTTP for Humans.', 'license': 'Apache-2.0', 'home_page': None, 'project_urls': {'Documentation': 'https://requests.readthedocs.io', 'Source': 'https://github.com/psf/requests'}}

Validated: requests v2.34.2 -- 'Python HTTP for Humans.'


### Task 3: `repair_json` (strip wrappers, parse)

The instinct almost everyone has first, and it is not a bad instinct: cheap, needs no
cooperation from the model, and genuinely fixes the most common problem.

What matters is being precise about its ceiling. It fixes everything *wrapped around*
the JSON. It fixes nothing wrong *inside* it. A `}` can appear inside a string value,
so the tempting non-greedy regex quietly truncates the object rather than failing loudly.

In [ ]:
def repair_json(raw: str):
    '''Pull a JSON object out of a model response that may be wrapped in noise.

    Strip markdown fences and any prose around them, then parse. Return the parsed dict, or
    None if what's left will not parse.

    Return None on failure rather than an empty dict. `{}` reads downstream as "the model
    returned a record with no fields", which is a different and much more confusing claim
    than "parsing failed" -- the same information-destroying move as returning a bare None
    from the validator earlier in this chapter.

    Watch the brace scan: a '}' can appear inside a string value, so find the LAST one, not
    the first.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


repair_json = check("ch07-json-repair", repair_json)

### Task 4: `retry_until_valid` (retry with budget)

Whether this works depends entirely on a property of the failure that the strategy
itself cannot see. If the malformation is **transient** (the model gets it right most
of the time and occasionally slips), retrying works. If the malformation is
**systematic** (this prompt, this schema, and this model produce the same broken shape
every time), re-prompting produces the identical failure and the budget buys nothing
but a bigger bill.

Second trap: `json.loads` succeeding is not the same as the contract being satisfied.
A model that cannot find a field in the source will sometimes leave it out and emit
flawless JSON without it.

In [ ]:
def retry_until_valid(model, max_attempts: int = 4):
    '''Re-prompt a model until its output is usable, or the budget runs out.

    Call model.generate() for a fresh response each attempt, and run it through repair_json.
    Return (record, attempts_used) on success, or (None, max_attempts) if the budget is
    exhausted. `attempts_used` is 1-based and must match the number of generations actually
    made -- it is the number a caller uses for cost accounting.

    Accept a response only if it parses AND carries every field in FIELDS. A parse alone is
    not the contract.

    max_attempts is a real ceiling, not a suggestion.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


retry_until_valid = check("ch07-retry-budget", retry_until_valid)

### Task 5: `constrained_decode` (grammar-masked generation)

The other two strategies let the model emit an invalid token and then deal with the
consequences. This one makes the invalid token unemittable.

At each step the decoder asks the model what it would consider emitting, discards every
candidate that could not appear at this position in a valid instance of the schema, and
takes the best of what survives. Because the mask is applied *before* anything is
committed, the output is well-formed by construction, and one generation is always
enough.

`is_allowed(emitted, token, fields)` and `is_complete(emitted, fields)` are provided.
Your job is the decode loop.

Two details are easy to get wrong: take the *highest-scoring* survivor, not the first
one (`candidates()` returns a distribution in no particular order). And stop when the
object is **complete**, not when the model runs out of things to say.

In [ ]:
from agentlib.structured_outputs import is_allowed, is_complete, FIELDS
print("Grammar functions loaded: is_allowed, is_complete")
print(f"Target fields: {FIELDS}")

In [ ]:
def constrained_decode(model, fields) -> dict:
    '''Decode one record from `model` under a grammar mask.

    Loop: reset the model, then repeatedly take model.candidates(), keep only the tokens for
    which is_allowed(emitted, token, fields) is true, commit the highest-scoring survivor with
    model.accept(token), and append it to what you've emitted. Stop as soon as
    is_complete(emitted, fields). Return the parsed dict.

    Raise ValueError if the mask ever leaves nothing to choose from -- that means the model
    and the grammar disagree about what is being generated, which is a real bug worth
    surfacing rather than a case to paper over.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


constrained_decode = check("ch07-constrained-decode", constrained_decode)

### The three strategies, side by side

Same model. Same record. Same failure modes. The only thing that varies is the strategy.

In [17]:
from agentlib.structured_outputs import is_allowed, is_complete

print(f"{'strategy':<22} {'result':<10} {'generations':>12}   notes")
print("-" * 78)

# 1. Repair only.
m = MockModel(RECORDS[0], quirks=QUIRKS)
raw = m.generate()
repaired = repair_json(raw)
print(f"{'repair afterwards':<22} {('OK' if repaired else 'FAILED'):<10} {m.generations:>12}   "
      f"{'unquoted key survives every trim' if not repaired else ''}")

# 2. Retry, against a model that is systematically broken.
m = MockModel(RECORDS[0], quirks=QUIRKS)
record, attempts = retry_until_valid(m, 4)
print(f"{'retry (systematic)':<22} {('OK' if record else 'FAILED'):<10} {m.generations:>12}   "
      f"{'four identical failures' if not record else ''}")

# 3. Retry, against a model that is only transiently broken.
m = MockModel(RECORDS[0], quirks=QUIRKS, flaky_until=2)
record, attempts = retry_until_valid(m, 4)
print(f"{'retry (transient)':<22} {('OK' if record else 'FAILED'):<10} {m.generations:>12}   "
      f"{'works -- at 3x the calls'}")

# 4. Constrained decoding, against the systematically broken model from row 2.
m = MockModel(RECORDS[0], quirks=QUIRKS)
decoded = constrained_decode(m, FIELDS)
print(f"{'constrained decoding':<22} {('OK' if decoded else 'FAILED'):<10} {m.generations:>12}   "
      f"first attempt, every time")
print()
print("raw text the constrained decoder actually emitted:")
print(" ", m.text)

strategy               result      generations   notes
------------------------------------------------------------------------------
repair afterwards      FAILED                1   unquoted key survives every trim
retry (systematic)     FAILED                4   four identical failures
retry (transient)      OK                    3   works -- at 3x the calls
constrained decoding   OK                    1   first attempt, every time

raw text the constrained decoder actually emitted:
  {"name":"numpy","version":"2.4.6","summary":"Fundamental package for array computing"}


### What that settles, and what it doesn't

The comparison above is the whole answer to "the model sometimes returns malformed JSON, what
is the most reliable fix", and it's worth being able to state the shape of it quickly:

- **Retrying** treats a systematic failure as if it were a flaky one. When the format is
  wrong because of how the model, prompt, and schema interact, every retry reproduces it.
- **Repairing** handles noise around the JSON and cannot touch broken syntax within it.
  Useful as a safety net, not as the primary mechanism.
- **Constraining the decoding** removes the failure mode instead of responding to it, which
  is why it is the right first answer whenever the API offers it.
- **A bigger model** makes the failure rarer, not impossible, and you are still writing the
  parser. Rare malformed JSON is arguably worse than frequent malformed JSON, because now it
  only breaks in production.
- **Raising temperature** is the opposite of a fix. If anything, lower it.

And the honest caveat, which is what separates a good answer from a recited one: a grammar
mask guarantees the output is **well-formed and on-schema**. It guarantees nothing about the
content being **correct**. The `_c5` case in the task above exists to make that concrete -- a
decoder that masks properly but picks carelessly returns `{"name": "nump", ...}`: perfect
JSON, right keys, wrong data. Schema-constrained decoding replaces your parser. It does not
replace your validation.

## Section 5: Playground

Experiments with the mock model's failure modes and the three strategies.

### Experiment 1: MockModel `flaky_until` parameter

The `flaky_until` parameter controls how many generations fail before the model starts
producing clean output. Try different values to see how retry count affects success.

In [ ]:
# --- EDIT THESE ---
# Try changing flaky_until to see how many retries are needed
from agentlib.structured_outputs import MockModel, RECORDS, QUIRKS, FIELDS

for flaky_n in [1, 3, 5]:
    m = MockModel(RECORDS[0], quirks=QUIRKS, flaky_until=flaky_n)
    record, attempts = retry_until_valid(m, max_attempts=8)
    status = "OK" if record else "FAILED"
    print(f"  flaky_until={flaky_n}: {status} after {attempts} attempts (budget=8)")

### Experiment 2: Which quirk types break JSON parsing?

Each quirk introduces a different kind of noise. Some are fixable by `repair_json`,
some are not. Try each one individually.

In [ ]:
# --- EDIT THESE ---
# Try different combinations of quirks
from agentlib.structured_outputs import MockModel, RECORDS, QUIRKS
import json as _json

quirk_sets = {
    "fence only": ["fence"],
    "preamble only": ["preamble"],
    "epilogue only": ["epilogue"],
    "trailing_comma": ["trailing_comma"],
    "unquoted_keys": ["unquoted_keys"],
    "fence + preamble": ["fence", "preamble"],
    "all quirks": QUIRKS,
}

print(f"{'quirks':<24s} {'raw parseable?':<16s} {'repair fixes it?'}")
print("-" * 60)
for label, quirks in quirk_sets.items():
    m = MockModel(RECORDS[0], quirks=quirks)
    raw = m.generate()
    try:
        _json.loads(raw)
        raw_ok = True
    except _json.JSONDecodeError:
        raw_ok = False
    repaired = repair_json(raw)
    print(f"{label:<24s} {str(raw_ok):<16s} {repaired is not None}")

### Experiment 3: Grammar mask in action

Watch which tokens get accepted and rejected at each step of constrained decoding.
The grammar mask rejects tokens that would produce invalid JSON.

In [ ]:
# --- EDIT THESE ---
# Try a different record index (0, 1, 2) to see how the mask adapts
RECORD_INDEX = 0

from agentlib.structured_outputs import MockModel, RECORDS, FIELDS, QUIRKS, is_allowed, is_complete

m = MockModel(RECORDS[RECORD_INDEX], quirks=QUIRKS)
m.reset()
emitted = ""
for step in range(50):
    candidates = m.candidates()
    allowed = [(tok, score) for tok, score in candidates if is_allowed(emitted, tok, FIELDS)]
    rejected = [(tok, score) for tok, score in candidates if not is_allowed(emitted, tok, FIELDS)]
    if not allowed:
        print(f"  step {step}: no allowed tokens, stopping")
        break
    best_tok, best_score = max(allowed, key=lambda x: x[1])
    m.accept(best_tok)
    emitted += best_tok
    if step < 15 or is_complete(emitted, FIELDS):
        rej_preview = ", ".join(f"{t!r}" for t, _ in rejected[:3])
        extra = "..." if len(rejected) > 3 else ""
        print(f"  step {step:2d}: accepted {best_tok!r:16s} rejected {len(rejected)} tokens ({rej_preview}{extra})")
    if is_complete(emitted, FIELDS):
        print(f"\nComplete after {step + 1} steps:")
        print(f"  {emitted}")
        break

### Experiment 4: Retry budget vs. success rate

How does the retry budget affect success rate when failures are transient vs. systematic?

In [ ]:
# --- EDIT THESE ---
# Try different budgets and flaky_until values
from agentlib.structured_outputs import MockModel, RECORDS, QUIRKS

budgets = [1, 3, 5, 10]
print(f"{'budget':<10s} {'systematic':<14s} {'transient (flaky_until=2)'}")
print("-" * 50)
for budget in budgets:
    m = MockModel(RECORDS[0], quirks=QUIRKS)
    record_s, attempts_s = retry_until_valid(m, max_attempts=budget)
    m = MockModel(RECORDS[0], quirks=QUIRKS, flaky_until=2)
    record_t, attempts_t = retry_until_valid(m, max_attempts=budget)
    print(f"{budget:<10d} {'OK' if record_s else 'FAILED':<14s} {'OK' if record_t else 'FAILED'} ({attempts_t} attempts)")

## Section 6: Break It

Four failure shapes, each injected into a copy of a real response. The base data is
genuinely real (fetched from the live MCP server); each failure shape is a deliberate,
clearly-labeled injection.

**Hint 1**: Check the error type first (timeout vs. parse error vs. wrong values).
A version mismatch looks like valid data with unexpected field names.

**Hint 2**: A `missing` error (field absent) means the upstream shape changed. A
`type` error (field present, wrong type) means the data is corrupted.

**Production impact**: a single generic `except Exception: log and move on` handler
silently maps all four failures to the same "try again or give up" path. Malformed data
gets retried against the same corrupted source. Semantically wrong data gets retried and
returns the same wrong answer. Version mismatches get retried forever.

**Interview follow-up**: "Your tool returns valid JSON with wrong values. How is that
different from a parse error, and why does the recovery strategy differ?"

### The real baseline response

Starting from a real, validated response from the MCP server. Each scenario below
injects exactly one failure type into a copy of it.

In [6]:
real_response = await call_get_package_info("numpy")
print("Real base response:", real_response)


Real base response: {'name': 'numpy', 'version': '2.5.2', 'summary': 'Fundamental package for array computing in Python', 'license': None, 'home_page': None, 'project_urls': {'documentation': 'https://numpy.org/doc/', 'download': 'https://pypi.org/project/numpy/#files', 'homepage': 'https://numpy.org', 'release notes': 'https://numpy.org/doc/stable/release', 'source': 'https://github.com/numpy/numpy', 'tracker': 'https://github.com/numpy/numpy/issues'}}


### 1. Transient: the call itself fails, not the data

In [7]:
class _FlakyCallCounter:
    '''Simulates a tool call that times out the first couple of attempts, then succeeds --
    the transient-failure shape from this chapter's concept table. The underlying call is
    still the real one; only whether it's allowed to complete is being controlled here.'''
    def __init__(self, fail_times: int):
        self.fail_times = fail_times
        self.attempts = 0

    async def call(self, package_name: str) -> dict:
        self.attempts += 1
        if self.attempts <= self.fail_times:
            raise TimeoutError(f"attempt {self.attempts}: simulated timeout")
        return await call_get_package_info(package_name)


async def call_with_retry(fn, *args, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return await fn(*args)
        except TimeoutError as e:
            if attempt == max_attempts:
                raise
            print(f"  attempt {attempt} failed ({e}), retrying...")
    raise RuntimeError("unreachable")


flaky = _FlakyCallCounter(fail_times=2)
result = await call_with_retry(flaky.call, "anthropic")
print(f"\nSucceeded on attempt {flaky.attempts}: {result['name']} v{result['version']}")


  attempt 1 failed (attempt 1: simulated timeout), retrying...
  attempt 2 failed (attempt 2: simulated timeout), retrying...



Succeeded on attempt 3: anthropic v0.121.0


### 2. Malformed: the response arrives, but doesn't type-check

A field has the wrong type: the tool responded, the shape is close, but something in it is
corrupt. Schema validation catches this; the fix is treating the same tool's next call with
suspicion (switch to a fresh, uncached read) rather than trusting the same corrupted source
again.

In [8]:
malformed_response = dict(real_response)
malformed_response["version"] = 2.5  # should be the string "2.5.2" -- a float slipped in

try:
    PackageInfo.model_validate(malformed_response)
    print("Validated cleanly -- this shouldn't happen for this scenario.")
except ValidationError as e:
    print("Schema validation caught it:")
    for err in e.errors():
        print(f"  field {err['loc']}: {err['type']} -- {err['msg']}")


Schema validation caught it:
  field ('version',): string_type -- Input should be a valid string


### 3. Semantically wrong: valid shape, wrong content

Every field type-checks. The response is simply about the wrong thing. No schema can catch
this, because nothing about the shape is broken. This needs a check against what was
actually asked for, not just against the schema.

In [9]:
wrong_content_response = dict(real_response)
wrong_content_response["name"] = "not-numpy"  # a different package's data, somehow

validated = PackageInfo.model_validate(wrong_content_response)  # passes schema validation fine
print(f"Schema validation: OK ({validated.name} v{validated.version})")
print(f"Requested 'numpy', got data for {validated.name!r} -- schema alone can't catch this.")


Schema validation: OK (not-numpy v2.5.2)
Requested 'numpy', got data for 'not-numpy' -- schema alone can't catch this.


### 4. Version mismatch: the shape itself changed

Not a type error on a present field: a field is missing entirely, the signature of an
upstream API that changed shape (a rename, a restructure) out from under code that still
expects the old one.

In [10]:
version_mismatch_response = dict(real_response)
version_mismatch_response["pkg_version"] = version_mismatch_response.pop("version")  # renamed upstream

try:
    PackageInfo.model_validate(version_mismatch_response)
    print("Validated cleanly -- this shouldn't happen for this scenario.")
except ValidationError as e:
    for err in e.errors():
        print(f"  field {err['loc']}: {err['type']} -- {err['msg']}")


  field ('version',): missing -- Field required


### The decision router

One function that classifies a raw tool response into the taxonomy above and picks the
matching response (retry, switch, or ask-user) instead of a single generic error
handler.

The line that matters most is the one between **malformed** and **semantically wrong**,
because it is the one a schema cannot draw for you. Malformed data failed to type-check,
and a retry against a fresh source may fix it. Semantically wrong data type-checked
perfectly and describes something other than what was asked for -- and will come back
identical every single time, no matter how many times you retry it.

In [ ]:
def classify_failure(package_name: str, response: dict, model) -> dict:
    '''Sort a raw tool response into this chapter's four categories.

    Return {"category": ..., "action": ..., "detail": ...} using exactly these pairings:

      "ok"                 -> "proceed"    validated, and it describes what was asked for
      "malformed"          -> "switch"     a field failed type validation; retry a fresh source
      "version_mismatch"   -> "ask-user"   a field is missing ENTIRELY; the upstream shape moved
      "semantically_wrong" -> "ask-user"   valid, well-typed, and about the wrong package

    Use validate_tool_output above. Tell missing from mistyped via the error's .errors() --
    a "missing" type means version_mismatch, anything else means malformed. Put something
    specific in `detail`: whoever reads the log should not have to go and look.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


classify_failure = check("ch07-failure-classifier", classify_failure)

In [12]:
scenarios = {
    "clean": real_response,
    "malformed": malformed_response,
    "semantically wrong": wrong_content_response,
    "version mismatch": version_mismatch_response,
}

for label, response in scenarios.items():
    verdict = classify_failure("numpy", response, PackageInfo)
    print(f"[{label:19s}] {verdict['category']:19s} -> {verdict['action']:9s} {verdict['detail']}")

[clean              ] ok                  -> proceed   numpy v2.5.2
[malformed          ] malformed           -> switch    field(s) ['version'] failed type validation
[semantically wrong ] semantically_wrong  -> ask-user  asked for 'numpy', got data for 'not-numpy'
[version mismatch   ] version_mismatch    -> ask-user  field(s) ['version'] missing entirely -- the upstream shape changed


## Section 7: Interview Q&A

### Question 1: "What problem does MCP solve that direct API integration does not?"

**Model answer**: Without MCP, every agent framework writes custom glue for every tool:
M frameworks times N tools equals M*N integrations. MCP standardizes the client-server
interface so any compliant client talks to any compliant server. The cost drops from M*N
to M+N. This is the same move HTTP made for web clients and servers. The client discovers
available tools at connection time through a structured handshake, calls them by name with
JSON arguments, and gets structured results back. Neither side needs to know the other's
internals.

### Question 2: "When would you use JSON repair vs. retry vs. constrained decoding?"

**Model answer**: Repair first, because it is cheap (one generation) and handles the most
common failure (prose and fences around valid JSON). It cannot fix broken syntax inside
the JSON itself. Retry next, but only for transient failures: if the same prompt and model
produce the same break every time, retrying wastes the budget. Constrained decoding is the
right first answer whenever the API offers it, because it removes the failure mode instead
of responding to it: the grammar mask guarantees well-formed, on-schema output in one
generation. The honest caveat: constrained decoding guarantees shape, not correctness.

### Question 3: "Your tool returns valid JSON but the values are wrong. What do you do?"

**Model answer**: This is the semantically wrong failure type. Schema validation passed,
so the problem is not structural. The response describes something other than what was
asked for (asked for "numpy", got data for "not-numpy"). Retrying cannot help because the
same query returns the same wrong data. The correct response is to check the returned data
against what was requested (does the `name` field match the query?) and escalate to a human
or verification step when it does not.

### Question 4: "How do you handle a tool that is intermittently unreachable?"

**Model answer**: This is the transient failure type. The request was fine; the environment
glitched (timeout, connection reset, rate limit). Retry with exponential backoff and jitter,
same as Chapter 4. The key distinction: confirm the failure is actually transient before
retrying. A 401 (bad credentials) retried 5 times is 5 identical failures. A timeout
retried with backoff has a good chance of succeeding. Check the error type, not just the
fact that an error occurred.

### Question 5: "Explain grammar-masked generation. When is it worth the complexity?"

**Model answer**: At each decoding step, the model proposes candidate tokens. The grammar
mask discards every candidate that could not appear at this position in a valid JSON
instance of the schema, then takes the highest-scoring survivor. Because the mask is
applied before anything is committed, the output is well-formed by construction. It is
worth the complexity whenever you need structured output from every request, not just most
of them. A retry loop that cuts errors from 4% to 0.9% still fails for one in a hundred
requests. Constrained decoding makes the failure rate zero for structural issues.

### Tool integration drill

Answer each of these on your own before checking the model answers.

In [18]:
from agentlib.self_check import drill as open_drill

drill = open_drill(7)
drill.questions()

Chapter 7 written drill — 8 questions

1. Definitional: why MCP standardizes the client-server interface, not a shared library
2. Cold diagnosis: schema-valid data, confidently wrong answer
3. Design judgment: "retry three times with backoff" as a universal policy
4. Judgment call: catching schema drift automatically
5. Rapid-fire: malformed JSON from the model
6. Cold diagnosis: the retry loop that helps everyone except one customer
7. Judgment call: does constrained output let you delete validation?
8. Design judgment: six fields, two frequently absent


#### Answering these

Write your answer into the slot for each question, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side. `check(n)` will
not show you an answer until you have written one of your own. If you want it anyway,
`drill.reveal(n)` is there and makes no judgement.

In [19]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

# Question 5
drill.attempt(5, '''
(Your answer here.)
''')

# Question 6
drill.attempt(6, '''
(Your answer here.)
''')

# Question 7
drill.attempt(7, '''
(Your answer here.)
''')

# Question 8
drill.attempt(8, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder
  5. not recorded — it is still the placeholder
  6. not recorded — it is still the placeholder
  7. not recorded — it is still the placeholder
  8. not recorded — it is still the placeholder

Chapter 7: 0/8 answered
  still open: [1, 2, 3, 4, 5, 6, 7, 8]


In [20]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  Definitional: why MCP standardizes the client-server interface, not a shared library

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References

1. Anthropic (2024). "Model Context Protocol specification."
   https://modelcontextprotocol.io/

2. Pydantic documentation. https://docs.pydantic.dev/

3. JSON Schema specification. https://json-schema.org/

4. OpenAI (2024). "Structured outputs." API documentation for `response_format`.

5. llama.cpp GBNF grammars. Grammar-constrained generation for local models.
   https://github.com/ggml-org/llama.cpp/blob/master/grammars/README.md

6. Outlines (dottxt). Regex-to-FSM compiler for structured generation.
   https://github.com/dottxt-ai/outlines

Related chapters:
- Chapter 4 (retry with backoff, circuit breaker patterns)
- Chapter 6 (tool security and prompt injection through tool inputs)
- Chapter 8 (system design: choosing tools and integration patterns)

## Next: Chapter 8, System Design and Engineering Judgment

This chapter was about what happens when tools go wrong: transient failures, malformed
responses, semantically wrong results, and version mismatches. Chapter 8 steps back from
any single technique to a repeatable framework for approaching an agent design problem
from scratch, including a question this course has been answering chapter by chapter:
when not to use an agent at all.